# IMT Controls — Slot Filling from Knowledge Graph
## Multi-Round Pipeline with 14-Slot Batches + Answerability Analysis

**Strategy:**
- The baseline has **26 controls / 108 slots** — too many to fill reliably in one shot.
- The professor's constraint: **use only 14 slots per LLM call** → run multiple rounds.
- After filling, we **analyse which questions the graph can answer** vs which it structurally cannot.

**Pipeline:**
1. Load the Knowledge Graph (JSON) and the Baseline CSV (108 slots)
2. Split slots into **batches of 14**
3. For each batch: call the LLM with the graph + 14 slots → fill values
4. Merge all rounds into a final filled baseline
5. **Answerability analysis**: classify each slot as `answerable` / `unanswerable` / `ambiguous`
6. Export results + analysis report


## Bloc 1 — Imports & Configuration

In [ ]:
import os
import json
import math
import pandas as pd
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA

load_dotenv()

print("Imports OK")
print(f"NVIDIA_API_KEY : {'OK' if os.getenv('NVIDIA_API_KEY') else 'MISSING'}")


## Bloc 2 — LLM Initialization

In [ ]:
llm = ChatNVIDIA(
    model="mistralai/mistral-large-3-675b-instruct-2512",
    api_key=os.getenv("NVIDIA_API_KEY"),
    temperature=0,
    max_completion_tokens=16000,
)
print("LLM ready")


## Bloc 3 — Load Knowledge Graph

In [ ]:
# ── Update this path to point to your graph JSON ──────────────────────────────
GRAPH_JSON_PATH = "SecuraOps High_graph_4pass.json"   # <-- change as needed

with open(GRAPH_JSON_PATH, "r", encoding="utf-8") as f:
    graph = json.load(f)

entities    = graph.get("entities", [])
relations   = graph.get("relationships", [])

print(f"Graph loaded: {len(entities)} entities, {len(relations)} relationships")

from collections import Counter
label_counts = Counter(e["label"] for e in entities)
print("\nNode types:")
for label, count in sorted(label_counts.items()):
    print(f"  {label:<20}: {count}")


## Bloc 4 — Load Baseline CSV (108 slots)

In [ ]:
BASELINE_CSV = "Baseline_pour_IMT_Controls_Slots_IMT_.csv"

df_baseline = pd.read_csv(BASELINE_CSV, sep=";", encoding="utf-8-sig")

# Normalise column name for description
df_baseline.rename(columns={
    [c for c in df_baseline.columns if "Description" in c][0]: "description"
}, inplace=True)

print(f"Baseline loaded: {len(df_baseline)} rows")
print(f"Controls: {df_baseline['Control_ID'].nunique()}")
print(f"Slot types: {df_baseline['Slot_Type'].value_counts().to_dict()}")
df_baseline.head(5)


## Bloc 5 — Build Compact Graph Summary for the LLM

In [ ]:
def build_graph_summary(entities: list, relationships: list, max_entities: int = 200) -> str:
    """
    Produce a compact text representation of the knowledge graph
    that fits comfortably within a single LLM call alongside 14 slots.
    
    If the graph is very large we truncate to max_entities but keep
    at least one representative entity per label.
    """
    # ── Entity block ──────────────────────────────────────────────────────────
    entity_lines = []
    seen_labels = set()
    sampled = []

    # First pass: keep at least 1 per label
    for e in entities:
        if e["label"] not in seen_labels:
            sampled.append(e)
            seen_labels.add(e["label"])

    # Second pass: fill up to max_entities
    added_ids = {id(e) for e in sampled}
    for e in entities:
        if len(sampled) >= max_entities:
            break
        if id(e) not in added_ids:
            sampled.append(e)
            added_ids.add(id(e))

    for e in sampled:
        props = ", ".join(f"{k}={json.dumps(v)}" for k, v in e.get("properties", {}).items())
        entity_lines.append(f"  [{e['label']}] {props}")

    # ── Relationship block ────────────────────────────────────────────────────
    rel_lines = []
    for r in relationships[:300]:   # cap to avoid overflow
        fk = list(r.get("from_key", {}).values())
        tk = list(r.get("to_key",   {}).values())
        from_name = fk[0] if fk else "?"
        to_name   = tk[0] if tk else "?"
        rel_lines.append(
            f"  ({r['from_label']}:{from_name}) -[{r['rel_type']}]-> ({r['to_label']}:{to_name})"
        )

    summary = "=== KNOWLEDGE GRAPH ENTITIES ===\n"
    summary += "\n".join(entity_lines)
    summary += "\n\n=== KNOWLEDGE GRAPH RELATIONSHIPS ===\n"
    summary += "\n".join(rel_lines)
    return summary


GRAPH_SUMMARY = build_graph_summary(entities, relations)
print(f"Graph summary: {len(GRAPH_SUMMARY):,} characters")
print("\nPreview (first 800 chars):")
print(GRAPH_SUMMARY[:800])


## Bloc 6 — Split Slots into Batches of 14

In [ ]:
BATCH_SIZE = 14   # ← professor constraint

def make_slot_batches(df: pd.DataFrame, batch_size: int = 14) -> list:
    """Split the baseline DataFrame into batches of `batch_size` slots."""
    all_slots = df.to_dict(orient="records")
    batches = []
    for i in range(0, len(all_slots), batch_size):
        batches.append(all_slots[i : i + batch_size])
    return batches


slot_batches = make_slot_batches(df_baseline, BATCH_SIZE)
n_batches    = len(slot_batches)

print(f"Total slots  : {len(df_baseline)}")
print(f"Batch size   : {BATCH_SIZE}")
print(f"Batches      : {n_batches}  ({n_batches} LLM calls needed)")
for i, b in enumerate(slot_batches):
    ids = [s['Control_ID'] for s in b]
    keys = [s['Slot_Key'] for s in b]
    print(f"  Batch {i+1:>2}: {len(b)} slots | controls {set(ids)} | {keys}")


## Bloc 7 — Prompt Builder for Each Batch

In [ ]:
def build_slot_prompt(graph_summary: str, batch: list) -> tuple[str, str]:
    """
    Build system + user prompt for filling one batch of ≤14 slots
    from the knowledge graph.

    Returns (system_prompt, user_prompt).
    """
    # Format slot definitions for the prompt
    slot_defs = []
    for slot in batch:
        cid   = slot["Control_ID"]
        cname = slot["Control_Name"]
        key   = slot["Slot_Key"]
        stype = str(slot.get("Slot_Type", "bool"))
        enum  = slot.get("Enum_Values")
        desc  = slot.get("description", "")

        if pd.isna(enum) if hasattr(enum, "__class__") and enum.__class__.__name__ == "float" else False:
            enum = None

        if stype == "bool":
            type_hint = "bool  → true | false"
        elif stype == "enum" and enum:
            type_hint = f"enum  → one of: {enum}"
        elif stype == "list[string]":
            type_hint = 'list[string]  → JSON array of strings, e.g. ["AES-256"]'
        else:
            type_hint = "bool  → true | false"

        slot_defs.append(
            f"- slot_key: \"{key}\"\n"
            f"  control: {cid} — {cname}\n"
            f"  type: {type_hint}\n"
            f"  instruction: {desc}"
        )

    slots_text = "\n\n".join(slot_defs)

    system_prompt = """You are a cybersecurity analyst filling a security assessment baseline.
You will be given a knowledge graph (entities + relationships extracted from supplier documents)
and a list of slots to fill. Each slot represents a specific security control question.

Rules:
- Answer ONLY from what is explicitly present in the knowledge graph.
- For bool slots: return true or false.
- For enum slots: return exactly one of the allowed values.
- For list[string] slots: return a JSON array of strings.
- If the graph does not contain enough information to determine the value, set:
    "value": null
    "answerability": "unanswerable"
- If the graph contains partial or ambiguous information, set:
    "answerability": "ambiguous"
- If the graph clearly supports the answer, set:
    "answerability": "answerable"
- Always include a short "evidence" field (max 1 sentence) quoting the graph node/property that justifies the answer.
- Return ONLY a valid JSON array, no markdown, no explanation.

Output format:
[
  {
    "slot_key": "<key>",
    "control_id": "<Control_ID>",
    "value": <true|false|"enum_value"|["item1"]|null>,
    "answerability": "answerable" | "ambiguous" | "unanswerable",
    "evidence": "<short justification or 'Not found in graph'>"
  },
  ...
]"""

    user_prompt = (
        f"KNOWLEDGE GRAPH:\n{graph_summary}\n\n"
        f"SLOTS TO FILL ({len(batch)} slots):\n\n{slots_text}\n\n"
        "Fill all slots from the graph above. Return only the JSON array."
    )

    return system_prompt, user_prompt


# Quick test: show the prompt for batch 1 (truncated)
sys_p, usr_p = build_slot_prompt(GRAPH_SUMMARY, slot_batches[0])
print(f"System prompt : {len(sys_p):,} chars")
print(f"User prompt   : {len(usr_p):,} chars")
print(f"TOTAL         : {len(sys_p)+len(usr_p):,} chars (~{(len(sys_p)+len(usr_p))//4:,} tokens estimated)")


## Bloc 8 — LLM Call + JSON Parser

In [ ]:
def call_llm(system_prompt: str, user_prompt: str) -> str:
    """Invoke the LLM and return raw string content."""
    messages = [
        ("system", system_prompt),
        ("human",  user_prompt),
    ]
    raw = llm.invoke(messages).content.strip()
    # Strip markdown code fences if present
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.strip()
    return raw


def parse_json_safe(raw: str, context: str = "") -> list | None:
    """Parse JSON with graceful error handling."""
    try:
        result = json.loads(raw)
        if isinstance(result, list):
            return result
        # sometimes model wraps in a dict
        if isinstance(result, dict) and "slots" in result:
            return result["slots"]
        return [result]
    except json.JSONDecodeError as e:
        print(f"  [JSON ERROR] {context}: {e}")
        print(f"  Raw (first 300): {raw[:300]}")
        return None


print("LLM helpers ready")


## Bloc 9 — Multi-Round Slot Filling Runner

In [ ]:
def run_all_rounds(
    graph_summary: str,
    slot_batches: list,
    verbose: bool = True
) -> list:
    """
    Iterate over all slot batches (rounds), call the LLM for each,
    and collect all filled slots.

    Returns a flat list of result dicts:
      {slot_key, control_id, value, answerability, evidence}
    """
    all_results = []
    n = len(slot_batches)

    for round_idx, batch in enumerate(slot_batches, start=1):
        batch_keys = [s["Slot_Key"] for s in batch]
        if verbose:
            print(f"\n{'─'*60}")
            print(f"Round {round_idx}/{n}  ({len(batch)} slots)")
            print(f"  Slots: {batch_keys}")

        sys_p, usr_p = build_slot_prompt(graph_summary, batch)

        try:
            raw = call_llm(sys_p, usr_p)
            results = parse_json_safe(raw, context=f"Round {round_idx}")

            if results is None:
                # Mark all slots in this batch as parse errors
                for slot in batch:
                    all_results.append({
                        "slot_key":      slot["Slot_Key"],
                        "control_id":    slot["Control_ID"],
                        "control_name":  slot["Control_Name"],
                        "value":         None,
                        "answerability": "error",
                        "evidence":      "JSON parse error — retry recommended"
                    })
                continue

            # Index results by slot_key for easy lookup
            result_map = {r.get("slot_key"): r for r in results}

            for slot in batch:
                key    = slot["Slot_Key"]
                result = result_map.get(key, {})
                filled = {
                    "slot_key":      key,
                    "control_id":    slot["Control_ID"],
                    "control_name":  slot["Control_Name"],
                    "slot_type":     slot.get("Slot_Type", "bool"),
                    "value":         result.get("value"),
                    "answerability": result.get("answerability", "unanswerable"),
                    "evidence":      result.get("evidence", "Not found in graph"),
                }
                all_results.append(filled)

                if verbose:
                    ans_icon = {"answerable": "✅", "ambiguous": "⚠️",
                                "unanswerable": "❌", "error": "💥"}.get(
                                filled["answerability"], "?")
                    print(f"  {ans_icon} {key:<40} = {str(filled['value']):<10}  ({filled['answerability']})")

        except Exception as e:
            print(f"  [ERROR] Round {round_idx}: {e}")
            for slot in batch:
                all_results.append({
                    "slot_key":      slot["Slot_Key"],
                    "control_id":    slot["Control_ID"],
                    "control_name":  slot["Control_Name"],
                    "value":         None,
                    "answerability": "error",
                    "evidence":      f"Exception: {str(e)}"
                })

    return all_results


print("Multi-round runner ready")


## Bloc 10 — ▶️ Execute All Rounds

In [ ]:
print(f"Starting multi-round slot filling")
print(f"  Graph: {len(entities)} entities, {len(relations)} relationships")
print(f"  Slots: {len(df_baseline)} total → {len(slot_batches)} rounds of ≤{BATCH_SIZE}")

filled_slots = run_all_rounds(GRAPH_SUMMARY, slot_batches, verbose=True)

print(f"\n{'='*60}")
print(f"Rounds complete. Total slots filled: {len(filled_slots)}")


## Bloc 11 — Build Results DataFrame

In [ ]:
df_results = pd.DataFrame(filled_slots)

# Summary counts
ans_counts = df_results["answerability"].value_counts()
print("\nAnswerability summary:")
print(ans_counts.to_string())
print()
print(f"  Answerable   : {ans_counts.get('answerable',   0):>3}  ({ans_counts.get('answerable',0)   /len(df_results)*100:.1f}%)")
print(f"  Ambiguous    : {ans_counts.get('ambiguous',    0):>3}  ({ans_counts.get('ambiguous',0)    /len(df_results)*100:.1f}%)")
print(f"  Unanswerable : {ans_counts.get('unanswerable', 0):>3}  ({ans_counts.get('unanswerable',0) /len(df_results)*100:.1f}%)")
print(f"  Errors       : {ans_counts.get('error',        0):>3}")

df_results.head(10)


## Bloc 12 — Answerability Analysis by Control

This section answers the professor's core question:
- **Which questions can the graph answer?**
- **Which style of questions can the graph NOT answer, and why?**


In [ ]:
# ── Per-control summary ────────────────────────────────────────────────────────
control_summary = (
    df_results
    .groupby(["control_id", "control_name"])["answerability"]
    .value_counts()
    .unstack(fill_value=0)
    .reset_index()
)

# Ensure all columns exist
for col in ["answerable", "ambiguous", "unanswerable", "error"]:
    if col not in control_summary.columns:
        control_summary[col] = 0

control_summary["total"] = (
    control_summary["answerable"] +
    control_summary["ambiguous"] +
    control_summary["unanswerable"] +
    control_summary.get("error", 0)
)
control_summary["coverage_%"] = (
    (control_summary["answerable"] + 0.5 * control_summary["ambiguous"])
    / control_summary["total"] * 100
).round(1)

control_summary = control_summary.sort_values("coverage_%", ascending=False)

print("Per-control coverage:")
print(control_summary[["control_id", "control_name", "answerable",
                         "ambiguous", "unanswerable", "coverage_%"]].to_string(index=False))


## Bloc 13 — Qualitative Analysis: Question Styles the Graph Can vs Cannot Answer

In [ ]:
# ── Group unanswerable slots to identify structural patterns ──────────────────
unanswerable_df = df_results[df_results["answerability"] == "unanswerable"].copy()
answerable_df   = df_results[df_results["answerability"] == "answerable"].copy()

print(f"Unanswerable slots: {len(unanswerable_df)}")
print(f"Answerable slots  : {len(answerable_df)}")

print("\n" + "="*70)
print("QUESTIONS THE GRAPH CAN ANSWER")
print("="*70)
print("""
The graph is built from text documents (policies, contracts, SLA docs).
It reliably captures:

  1. EXISTENCE questions (bool)
     → "Does a CISO role exist?"      → GOV-02.ciso_named
     → "Is MFA implemented?"          → IAM-01.mfa_present
     → "Is an IRP documented?"        → OPS-01.irp_present
     The graph has 'Roles', 'Control', 'Framework' nodes with named properties.

  2. RELATIONSHIP questions
     → "Which controls are implemented by which technologies?"
     → "Which framework requires which control?"
     → "Which role executes which control?"
     Graph relationships (EXECUTES, IMPLEMENTED_BY, required_by) answer these directly.

  3. NAMED-ENTITY questions
     → "What encryption algorithms are used?"     → DATA-02.algorithms
     → "What is the TLS minimum version?"         → DATA-01.tls_min_version
     Algorithm, Protocol, Technology nodes carry named properties.

  4. SCOPE / ENUM questions when scope is documented
     → "What is the EDR scope?"      → ENDP-01.edr_scope
     → "What is the MFA scope?"      → IAM-01.mfa_scope
     Asset and Claim nodes carry 'scope' fields extracted from text.
""")

print("="*70)
print("QUESTIONS THE GRAPH CANNOT ANSWER (structural gaps)")
print("="*70)
print("""
  1. OPERATIONAL/METRIC questions
     → "Is monitoring 24/7?"          → OPS-02.monitoring_24_7
     → "Are phishing simulations continuous?" → TRAIN-03.phishing_continuous
     → "Is key rotation performed?"   → DATA-02.key_rotation
     WHY: The graph stores WHAT exists (nodes), not HOW it operates (operational cadence).
     Frequency, scheduling, and SLA metrics are rarely captured as named nodes.

  2. NEGATIVE EVIDENCE questions
     → "Is NDA the ONLY clause used?" → ACH-02.nda_only
     → "Is direct admin access allowed?" → NET-03.admin_direct_access_allowed
     WHY: The graph captures positive assertions. Absence of a node ≠ confirmed absence.
     Negative controls require document-level NLP, not graph traversal.

  3. PROCEDURAL DETAIL questions
     → "Are lessons learned documented after exercises?" → RES-02.lessons_learned
     → "Is there a break-glass procedure?"              → IAM-03.break_glass
     WHY: These are process maturity indicators buried in procedural text,
     not typically captured as discrete named entities in the schema.

  4. IMPLICIT SCOPE QUESTIONS (partial enum)
     → "Is FDE on all workstations or partial?"  → ENDP-02.fde_scope
     → "What is the JML automation level?"       → IAM-02.jml_automation_level
     WHY: The graph schema lacks enum-typed property fields. Scope/level
     information would need dedicated Claim.value + Claim.scope nodes to be reliable.

  5. TECHNICAL PROTOCOL DETAILS not in documents
     → "Is HSTS enforced?"      → DATA-01.hsts
     → "Is PFS enforced?"       → DATA-01.pfs
     → "Is SPF published?"      → DATA-03.spf / dkim / dmarc_policy
     WHY: These are infrastructure-level configurations not described in
     governance/policy documents — the source documents don't mention them.
""")


## Bloc 14 — Detailed View: Unanswerable Slots

In [ ]:
cols = ["control_id", "control_name", "slot_key", "slot_type", "answerability", "evidence"]
print("All unanswerable / ambiguous slots:")
print(
    df_results[df_results["answerability"].isin(["unanswerable", "ambiguous"])]
    [cols]
    .sort_values(["control_id", "answerability"])
    .to_string(index=False)
)


## Bloc 15 — Automatic Pattern Classification of Unanswerable Slots

In [ ]:
# Simple keyword-based classifier for unanswerable reason
PATTERNS = {
    "operational_cadence":   ["24_7", "continuous", "regular", "rotation", "periodic", "recurring", "frequency"],
    "negative_evidence":     ["only", "direct_access", "nda_only", "materials_only"],
    "procedural_maturity":   ["lessons", "break_glass", "tested", "exercised", "remediation", "reporting_mechanism"],
    "implicit_scope_enum":   ["scope", "automation_level", "fde_scope", "jml_automation"],
    "infra_config_not_docs": ["hsts", "pfs", "spf", "dkim", "dmarc", "rua", "ruf", "alignment"],
}

def classify_pattern(slot_key: str) -> str:
    sk = slot_key.lower()
    for pattern, keywords in PATTERNS.items():
        if any(kw in sk for kw in keywords):
            return pattern
    return "other"


df_results["pattern"] = df_results["slot_key"].apply(classify_pattern)

# Show pattern breakdown for unanswerable slots
pattern_counts = (
    df_results[df_results["answerability"] == "unanswerable"]
    ["pattern"].value_counts()
)

print("Unanswerable slots by structural pattern:")
print(pattern_counts.to_string())
print()
print("Example slots per pattern:")
for pattern in pattern_counts.index:
    examples = df_results[
        (df_results["answerability"] == "unanswerable") &
        (df_results["pattern"] == pattern)
    ]["slot_key"].head(4).tolist()
    print(f"  {pattern:<30}: {examples}")


## Bloc 16 — Export Results

In [ ]:
# ── CSV: full filled baseline ──────────────────────────────────────────────────
out_csv = "imt_controls_filled.csv"
df_results.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"Saved: {out_csv}")

# ── JSON: machine-readable ────────────────────────────────────────────────────
out_json = "imt_controls_filled.json"
with open(out_json, "w", encoding="utf-8") as f:
    json.dump(filled_slots, f, indent=2, ensure_ascii=False)
print(f"Saved: {out_json}")

# ── Analysis report (text) ────────────────────────────────────────────────────
out_report = "answerability_report.txt"
ans_counts = df_results["answerability"].value_counts()
with open(out_report, "w", encoding="utf-8") as f:
    f.write("IMT CONTROLS — ANSWERABILITY REPORT\n")
    f.write("="*60 + "\n\n")
    f.write(f"Total slots   : {len(df_results)}\n")
    f.write(f"Answerable    : {ans_counts.get('answerable',0)} ({ans_counts.get('answerable',0)/len(df_results)*100:.1f}%)\n")
    f.write(f"Ambiguous     : {ans_counts.get('ambiguous',0)} ({ans_counts.get('ambiguous',0)/len(df_results)*100:.1f}%)\n")
    f.write(f"Unanswerable  : {ans_counts.get('unanswerable',0)} ({ans_counts.get('unanswerable',0)/len(df_results)*100:.1f}%)\n")
    f.write(f"Errors        : {ans_counts.get('error',0)}\n\n")

    f.write("PER CONTROL COVERAGE\n" + "-"*40 + "\n")
    f.write(control_summary[["control_id","control_name","answerable","ambiguous","unanswerable","coverage_%"]]
            .to_string(index=False))
    f.write("\n\nUNANSWERABLE PATTERNS\n" + "-"*40 + "\n")
    f.write(pattern_counts.to_string())
    f.write("\n")
print(f"Saved: {out_report}")


## Bloc 17 — Visual Summary (matplotlib)

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("IMT Controls — Slot Filling Answerability Analysis", fontsize=14, fontweight="bold")

# ── Plot 1: Overall pie ───────────────────────────────────────────────────────
ax1 = axes[0]
colors = {"answerable": "#4caf50", "ambiguous": "#ff9800", "unanswerable": "#f44336", "error": "#9e9e9e"}
sizes  = [ans_counts.get(k, 0) for k in colors]
labels = [f"{k.capitalize()} ({v})" for k, v in zip(colors, sizes)]
ax1.pie(sizes, labels=labels, colors=list(colors.values()),
        autopct="%1.1f%%", startangle=140)
ax1.set_title("Overall Answerability (108 slots)")

# ── Plot 2: Per-control bar ───────────────────────────────────────────────────
ax2 = axes[1]
x    = range(len(control_summary))
w    = 0.6
ax2.bar(x, control_summary["answerable"],   color="#4caf50", label="Answerable",   width=w)
ax2.bar(x, control_summary["ambiguous"],    bottom=control_summary["answerable"],
        color="#ff9800", label="Ambiguous",   width=w)
ax2.bar(x, control_summary["unanswerable"],
        bottom=control_summary["answerable"] + control_summary["ambiguous"],
        color="#f44336", label="Unanswerable", width=w)
ax2.set_xticks(list(x))
ax2.set_xticklabels(control_summary["control_id"], rotation=90, fontsize=7)
ax2.set_title("Slots per Control")
ax2.set_ylabel("# Slots")
ax2.legend(fontsize=8)

# ── Plot 3: Unanswerable patterns ────────────────────────────────────────────
ax3 = axes[2]
pattern_counts_plot = df_results[df_results["answerability"] == "unanswerable"]["pattern"].value_counts()
ax3.barh(pattern_counts_plot.index, pattern_counts_plot.values, color="#f44336")
ax3.set_title("Why Slots Are Unanswerable\n(structural pattern)")
ax3.set_xlabel("# Slots")

plt.tight_layout()
plt.savefig("answerability_analysis.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: answerability_analysis.png")


## Bloc 18 — Final Summary

In [ ]:
print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"  Graph: {len(entities)} entities, {len(relations)} relationships")
print(f"  Slots: {len(df_results)} across {df_results['control_id'].nunique()} controls")
print(f"  Rounds: {len(slot_batches)} (batch size = {BATCH_SIZE})")
print()
ans = df_results['answerability'].value_counts()
print(f"  ✅ Answerable   : {ans.get('answerable',   0):>3} ({ans.get('answerable',0)   /len(df_results)*100:.0f}%)")
print(f"  ⚠️  Ambiguous    : {ans.get('ambiguous',    0):>3} ({ans.get('ambiguous',0)    /len(df_results)*100:.0f}%)")
print(f"  ❌ Unanswerable : {ans.get('unanswerable', 0):>3} ({ans.get('unanswerable',0) /len(df_results)*100:.0f}%)")
print()
print("  Outputs written:")
print(f"    imt_controls_filled.csv")
print(f"    imt_controls_filled.json")
print(f"    answerability_report.txt")
print(f"    answerability_analysis.png")
